In [1]:
import pandas as pd
import json
import torch
from constants import SKIP, TEST_PIECE_IDS, TO_IGNORE_INDICES

### training set

In [2]:
data_dir = "/mnt/ssd/hbli/datasets/PM2S_dataset/midi2scoretransformer/"
data_real = pd.read_csv(data_dir + "/ACPAS-dataset/metadata_R.csv")
data_synthetic = pd.read_csv(data_dir + "/ACPAS-dataset/metadata_S.csv")
asap_annotations = json.load(
    open(data_dir + "/asap-dataset/asap_annotations.json")
)
UNALIGNED = set(
    "{ASAP}/" + k
    for k, v in asap_annotations.items()
    if not v["score_and_performance_aligned"]
)
# Filter
data = pd.concat([data_real, data_synthetic])
# Initial filtering
data = data[(data["source"] == "ASAP") & data["aligned"]]
data = data[~data["performance_MIDI_external"].isin(SKIP)]
data = data[~data["performance_MIDI_external"].isin(UNALIGNED)]
data = data.drop_duplicates(subset=["performance_MIDI_external"])
# Filter by annotations
data.reset_index(inplace=True)
data.drop(TO_IGNORE_INDICES, inplace=True)

In [3]:
test_idx = data["piece_id"].isin(TEST_PIECE_IDS)

In [4]:
len(data), data['piece_id'].nunique()

(967, 206)

In [5]:
data_train = data[(data["piece_id"] % 10 != 0) & (~data["piece_id"].isin(TEST_PIECE_IDS))]
data_val = data[(data["piece_id"] % 10 == 0) & (~data["piece_id"].isin(TEST_PIECE_IDS))]
data_test = data[test_idx]

In [6]:
len(data_train), len(data_val), len(data_test)

(822, 86, 59)

In [ ]:
data_train['piece_id'].nunique(), data_val['piece_id'].nunique(), data_test['piece_id'].nunique()

(176, 16, 14)

In [17]:
os.path.dirname(os.path.dirname(os.path.abspath('/home/hbli/PM2S/MIDI2ScoreTransformer/midi2scoretransformer/eda.ipynb')))

'/home/hbli/PM2S/MIDI2ScoreTransformer'

### input and output stream

In [3]:
input_s, output_s = torch.load("/mnt/ssd/hbli/datasets/PM2S_dataset/midi2scoretransformer/cache/ffd0c19d7e74bf26822aa067290eaf49f899e4e59e470e514def2b947cc8341f.pkl", weights_only=False)

In [5]:
input_s['onset'].shape

torch.Size([1295])

In [29]:
output_s['offset'][:100]

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 1.0000, 1.0000, 1.0000, 1.5000,
        1.5000, 1.5000, 2.0000, 2.5000, 2.5000, 2.5000, 0.0000, 0.0000, 0.5000,
        1.0000, 1.0000, 1.5000, 1.5000, 1.5000, 1.5000, 2.5000, 0.0000, 0.0000,
        0.5000, 1.0000, 1.0000, 1.5000, 1.5000, 1.5000, 2.0000, 2.5000, 2.5000,
        0.0000, 0.0000, 0.0000, 0.5000, 1.0000, 1.0000, 1.5000, 1.5000, 2.0000,
        2.0000, 2.0000, 2.5000, 2.5000, 2.5000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.5000, 1.0000, 1.0000, 1.0000, 1.5000, 1.5000, 1.5000, 2.0000, 2.5000,
        2.5000, 2.5000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.5000, 1.5000, 1.5000, 1.5000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.5000, 1.0000, 1.0000, 1.0000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 2.0000, 2.5000, 2.5000, 2.5000, 2.5000, 2.5000, 0.0000,
        0.0000])

In [29]:
output_s['keysignature'].unique()

tensor([10.])

In [34]:
from music21 import key
key.KeySignature(14 - 7).transpose('d5')

<music21.key.KeySignature of 1 sharp>

In [ ]:
length = torch.FloatTensor([ 1435.,  1438.,  1429.,  1431.,  1433.,  1438.,  1440.,  1431.,  1430.,
          739.,   738.,   734.,   738.,   736.,   740.,  1400.,  1345.,  1361.,
         1340.,  1358.,  1739.,  1750.,  1766.,  1740.,  1762.,  1744.,   868.,
          792.,   799.,   803.,  1188.,  2456.,  2402.,   747.,   739.,   872.,
         1070.,  1049.,  1092.,   867.,   872.,   867.,   806.,   799.,   691.,
          704.,   984.,   975.,  1452.,  1450.,  1463.,  1443.,  1544.,   745.,
         1676.,  1659.,  1662.,  1671.,  1588.,   768.,  1760.,  1305.,  1310.,
          992.,  1020.,  1031.,   548.,   811.,   816.,   810.,   810.,   810.,
          811.,   806.,   811.,   808.,   459.,   444.,   443.,   444.,   465.,
          549.,   566.,   556.,   557.,   526.,   559.,   439.,   486.,   616.,
          608.,   612.,   613.,   611.,   608.,   675.,   562.,   564.,   564.,
          611.,   612.,   613.,   610.,   769.,   777.,   834.,   834.,   833.,
         1358.,  1574.,  1551.,  1103.,  1099.,  1104.,   920.,   924.,  1240.,
         1243.,   892.,   888.,   908.,   524.,   517.,   516.,   525.,   681.,
          785.,   782.,   780.,   786.,   789.,   795.,  1180.,  1185.,  1084.,
         1086.,   876.,   885.,   895.,  3811.,  3841.,  3881.,  3886.,  3194.,
         2787.,  2761.,  2804.,  2831.,  2822.,  1692.,  4315.,  3264.,  3319.,
         3260.,  3239.,  3213.,   494.,  7253.,  7148.,  7311.,  3112.,  2834.,
         6361.,  6365.,  6189.,  5969.,  6241.,  6304.,  2785.,  4217.,  2133.,
         2722.,   861.,   868.,  5280.,  2646.,  2628.,  2625.,  3990.,  1285.,
         1301.,  1295.,  4347.,  4279.,  4368.,  4346.,  1477.,  1507.,  1504.,
         2903.,  2863.,  2846.,  2921.,  2888.,  2880.,  2896.,  2950.,  1364.,
         3843.,  3834.,  3856.,  3836.,  3822.,  5531.,  5311.,  5443.,  4011.,
         4020.,  4000.,  3947.,  2111.,  2113.,  3520.,  3482.,  3474.,  3541.,
         3468.,  3545.,  2141.,  2134.,  4421.,  2424.,  2660.,  1798.,  4195.,
         4348.,  4055.,  4570.,  4577.,  6153.,  6290.,  6015.,  6247.,  6019.,
         6010.,  5963.,  4700.,  4771.,  4634.,  4773.,  4724.,  1703.,  1686.,
         1728.,  1716.,  1691.,  1422.,  1361.,  1369.,  1460.,  1386.,  1410.,
         1379.,  2232.,  1391.,  1375.,  2239.,  2254.,  2367.,  2230.,  2306.,
         2271.,  2235.,  2183.,  2283.,  1618.,  1607.,  1650.,  2351.,  2327.,
         2330.,  2339.,  2331.,  2239.,  2316.,  3615.,  3527.,  3386.,  3255.,
         3364.,  3296.,  3214.,  3259.,  3241.,  3366.,  3347.,  3216.,  3233.,
         2169.,  6277.,  6146.,  5087.,  4826.,  6128.,  6012.,  5864.,  5896.,
         5916.,  5905.,  6111.,  5864.,  5821.,  5982.,  5126.,  5248.,  1691.,
         4797.,  5053.,  4870.,  4841.,  3203.,  2009.,  3163.,  2725.,  2706.,
         1991.,  1987.,  2824.,  2822.,  3486.,  3484.,  3516.,  3499.,  3510.,
         3496.,  3494.,  7320.,  7245.,  7034.,  3297.,  3369.,  3126.,  3097.,
         4085.,  4154.,  4249.,  4133.,  4011.,  4187.,  4328.,  4139.,  4097.,
         4533.,  4743.,  4914.,  5337.,  9992., 10366., 10351.,  4095.,  4057.,
         4169.,  1911.,  4243.,  4304.,  4215.,  4274.,  4395.,  3972.,  3975.,
         2955.,  2941.,  1248.,  1430.,  3151.,  3145.,  3151.,  3153.,  3116.,
         3122.,  4282.,  4239.,  4310.,  4425.,  5935.,  5978.,  2634.,  1034.,
         2363.,  2412.,  2372.,  3364.,  3288.,  3030.,  3067.,  3109., 15733.,
        15840.,  6173.,  8266.,  7629.,  8071.,  7910.,  7548.,  1353.,  2398.,
          946.,   982.,  1428.,  1642.,  1662.,  1221.,  1196.,  1171.,   637.,
         1358.,  1237.,   888.,   510.,   523.,  1692.,  3865.,  3814.,  3825.,
         1954.,  3324.,  3300.,  4107.,  4912.,  3914.,  3944.,  3912.,  2736.,
         2826.,  2801.,  2891.,  2611.,  2864.,  2748.,  3242.,  3220.,  3208.,
         3212.,  3023.,  2222.,  2214.,  2222.,  1490.,  3069.,  3062.,  3047.,
          490.,   492.,  2978.,  3092.,  6249.,  6349.,  6392.,  6381.,  6236.,
         6391.,  6400.,  6496.,  6408.,  6314.,  6347.,  4231.,  4255.,  2787.,
         2827.,  2830.,   873.,   874.,   864.,  3814.,  5299.,  3910.,  2628.,
         2636.,  2595.,  2635.,  1275.,  1290.,  1902.,  5150.,  4344.,  4303.,
         4359.,  4362.,  1516.,  1511.,  1509.,  1509.,  1512.,  1499.,  2912.,
         2879.,  2921.,  2901.,  2931.,  2849.,  2920.,  2908.,  2833.,  2912.,
         2880.,  2941.,  2914.,  1362.,  1374.,  1345.,  1160.,  1356.,  1373.,
         3802.,  3873.,  3815.,  3830.,  3828.,  4113.,  4111.,  5477.,  5723.,
         4147.,  4108.,  4053.,  4029.,  4031.,  4037.,  4009.,  2114.,  3526.,
         3482.,  3463.,  2076.,  2064.,  1478.,  1480.,  4443.,  1641.,  4276.,
         4338.,  4397.,  6265.,  6239.,  6105.,  6231.,  6035.,  4776.,  4787.,
         4607.,  4771.,  1360.,  1522.,  1406.,  1406.,  1365.,  1402.,  1385.,
         1378.,  1359.,  1458.,  1430.,  1418.,  1409.,  1382.,  1427.,  1385.,
         1421.,  2251.,  2264.,  2240.,  1436.,  1418.,  1366.,  1467.,  1367.,
         1373.,  1359.,  1408.,  1322.,  1931.,  2272.,  2264.,  2284.,  2204.,
         2231.,  2238.,  2212.,  2238.,  2244.,  2228.,  2218.,  2230.,  2222.,
         1659.,  1620.,  1622.,  1615.,  1617.,  1620.,  1671.,  1552.,  1717.,
         1765.,  1788.,  2345.,  2328.,  2356.,  2318.,  2333.,  2354.,  2304.,
         2331.,  2327.,  2319.,  2306.,  2324.,  2356.,  2365.,  2311.,  2320.,
         2351.,  2321.,  2352.,  2264.,  2343.,  3638.,  3653.,  3657.,  3648.,
         3701.,  3738.,  3544.,  3504.,  3523.,  3218.,  3272.,  3506.,  3248.,
         3336.,  3286.,  3255.,  3319.,  3191.,  3299.,  3212.,  3330.,  3218.,
         3312.,  1210.,  1476.,  1443.,  2188.,  1516.,  1583.,  1471.,  1455.,
         5163.,  5989.,  5998.,  4171.,  4144.,  4005.,  3958.,  2139.,  1975.,
         1737.,  1736.,  1400.,  1398.,  1413.,  1411.,  1436.,  1425.,  1388.,
         1404.,  1401.,  1847.,  1853.,  1876.,  1881.,  1862.,  1874.,  1857.,
         1869.,  1838.,  4846.,  4933.,  4831.,  4906.,  4821.,  2105.,  2129.,
         1531.,  1500.,  1503.,  2037.,  2391.,  2715.,  2700.,  2741.,  2652.,
         2635.,  2642.,  3480.,  3497.,  3554.,  3497.,  3523.,  3526.,  3492.,
         3510.,  3495.,  3490.,  3491.,  2342.,  6943.,  7025.,  7270.,  3340.,
         3263.,  3426.,  3382.,  3074.,  2905.,  4133.,  4103.,  4109.,  4115.,
         4032.,  4091.,  4167.,  4249.,  4088.,  4042.,  4307.,  4263.,  4167.,
         4016.,  4232.,  4121.,  4203.,  4118.,  4355.,  4895.,  4965.,  4909.,
         5247.,  4965.,  9643., 10098., 10104., 10284., 10038., 10088.,  9882.,
        10003., 10108., 10134.,  4454.,  4252.,  4096.,  4142.,  4311.,  4426.,
         4220.,  4047.,  4299.,  4181.,  4435.,  4188.,  4142.,  4414.,  4290.,
         4308.,  4454.,  4466.,  5141.,  4956.,  5012.,  2821.,  1386.,  1377.,
         2947.,  3278.,  3314.,  3295.,  3306.,  3328.,  2688.,  1463.,  4125.,
         4361.,  4282.,  4099.,  3865.,  4266.,  4069.,  4196.,  1789.,  1766.,
         3178.,  5434.,  5932.,  2418.,  1321.,  1378.,  3541.,  3280.,  3325.,
         3010.,  3057.,  2926., 15979.,  3461.,  3384.,   674.,   656.,   701.,
          712.,   711.,  1531.,  1502.,  1489.,  6328.,  6410.,  6212.,  8043.,
         8088.,  7892.,  7631.])

In [22]:
torch.multinomial(length, 1, replacement=True)

tensor([389])

In [ ]:
length

### PDMX data EDA

In [ ]:
df_pdmx = pd.read_csv("/mnt/ssd/hbli/datasets/PDMX/PDMX.csv")

In [ ]:
df_pdmx[df_pdmx['subset:no_license_conflict'] & df_pdmx['subset:valid_mxl_pdf']]

In [ ]:
df_pdmx[(df_pdmx['title'].str.contains('Fugue') == True) & (df_pdmx['title'].str.contains('870') == True)][['title', 'composer_name', 'tracks']]

In [ ]:
df_pdmx['mxl'][0]

In [ ]:
df_pdmx['subset:valid_mxl_pdf']

## musicrender vs. parse_mxl

In [ ]:
def build_parse_mxl_surrogate_x(mxl_path: str, surrogate_qpm: float = 120.0):
    """
    Build the current mechanical surrogate x from parse_mxl output,
    using the same logic as dataset_v4:
      - pitch from cont["pitch"]
      - onset from cont["absolute_onset"] converted to pseudo-MIDI seconds
      - duration from cont["duration"] converted to pseudo-MIDI seconds
      - onset rebased so the first note starts at 0
    """
    cont = MultistreamTokenizer.parse_mxl(mxl_path)

    pitch = cont["pitch"].clone().long()
    onset = cont["absolute_onset"].clone().float()
    duration = cont["duration"].clone().float()

    sec_per_quarter = 60.0 / surrogate_qpm
    onset_sec = onset * sec_per_quarter
    duration_sec = duration * sec_per_quarter

    if len(onset_sec) > 0:
        onset_sec = onset_sec - onset_sec[0]

    return {
        "pitch": pitch,
        "onset": onset_sec,
        "duration": duration_sec,
    }

In [ ]:
def build_musicrender_midi_x(midi_path: str):
    """
    Build x from MusicRender-rendered MIDI.
    This uses the same parse_midi() logic as the real paired MIDI input branch.
    """
    cont = MultistreamTokenizer.parse_midi(midi_path)
    return {
        "pitch": cont["pitch"].clone().long(),
        "onset": cont["onset"].clone().float(),
        "duration": cont["duration"].clone().float(),
    }

In [ ]:
xml_x = build_parse_mxl_surrogate_x("/mnt/ssd/hbli/datasets/PDMX/mxl/1/11/QmbbFEQzNihEnR2EvTumtWeYgcCWcqUvEeCWPCk5GZA7GQ.mxl")
mr_x = build_musicrender_midi_x("/mnt/ssd/hbli/datasets/PDMX/render/musicrender_midi.mid")

print("parse_mxl surrogate lengths:")
print({k: len(v) for k, v in xml_x.items()})

print("\nMusicRender MIDI lengths:")
print({k: len(v) for k, v in mr_x.items()})

In [ ]:
def to_df(x_dict, name: str):
    return pd.DataFrame({
        f"{name}_idx": range(len(x_dict["pitch"])),
        f"{name}_pitch": x_dict["pitch"].tolist(),
        f"{name}_onset": x_dict["onset"].tolist(),
        f"{name}_duration": x_dict["duration"].tolist(),
    })

xml_df = to_df(xml_x, "xml")
mr_df = to_df(mr_x, "mr")

display(xml_df.head(10))
display(mr_df.head(10))